# 18 — Sprint 4: análise comercial integrada

Este é o ponto de entrada da solução. Ele carrega os módulos 12 a 17, recebe uma transcrição por vez e devolve produto, sentimento, risco de churn, oportunidade, termos e recomendação em um único JSON.

## 1. Carregar os módulos

Os notebooks anteriores compartilham o mesmo kernel. Essa composição mantém as funcionalidades separadas sem criar uma biblioteca ou CLI paralela.

In [ ]:
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd().resolve()
if not (NOTEBOOKS_DIR / '12_sprint4_foundation.ipynb').exists():
    NOTEBOOKS_DIR = NOTEBOOKS_DIR / 'notebooks'
if not (NOTEBOOKS_DIR / '12_sprint4_foundation.ipynb').exists():
    raise FileNotFoundError('Execute o notebook a partir da raiz ou da pasta notebooks.')

for module_name in [
    '12_sprint4_foundation.ipynb',
    '13_sprint4_products_and_terms.ipynb',
    '14_sprint4_sentiment.ipynb',
    '15_sprint4_churn.ipynb',
    '16_sprint4_opportunity.ipynb',
    '17_sprint4_recommendation.ipynb',
]:
    get_ipython().run_line_magic('run', f'"{NOTEBOOKS_DIR / module_name}"')


## 2. Carregar exatamente uma transcrição

`carregar_transcricao` aceita texto direto ou um arquivo, nunca os dois ao mesmo tempo. Arquivos `.txt` preservam o conteúdo original; arquivos `.json` devem conter um objeto e o campo configurado. Listas são rejeitadas para impedir processamento em lote acidental.

In [ ]:
def _validar_texto_transcricao(valor: Any, origem: str) -> str:
    if not isinstance(valor, str):
        raise TypeError(f"A transcrição de {origem} deve ser uma string.")
    if not valor.strip():
        raise ValueError(f"A transcrição de {origem} não pode estar vazia.")
    return valor


def carregar_transcricao(
    *,
    texto: str | None = None,
    arquivo: str | Path | None = None,
    campo_json: str = "transcricao",
) -> str:
    """Carrega uma única transcrição sem alterar seu conteúdo."""
    if (texto is None) == (arquivo is None):
        raise ValueError("Informe exatamente uma fonte: texto ou arquivo.")
    if texto is not None:
        return _validar_texto_transcricao(texto, "texto direto")

    caminho = Path(arquivo)
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de transcrição não encontrado: {caminho}")
    if caminho.suffix.casefold() == ".txt":
        with caminho.open("r", encoding="utf-8", newline="") as stream:
            valor = stream.read()
    elif caminho.suffix.casefold() == ".json":
        try:
            payload = json.loads(caminho.read_text(encoding="utf-8"))
        except json.JSONDecodeError as error:
            raise ValueError(f"JSON inválido em {caminho.name}.") from error
        if not isinstance(payload, dict):
            raise ValueError("A entrada deve ser um objeto JSON com uma transcrição.")
        if campo_json not in payload:
            raise ValueError(f"O JSON não contém o campo {campo_json!r}.")
        valor = payload[campo_json]
    else:
        raise ValueError("Use um arquivo com extensão .txt ou .json.")
    return _validar_texto_transcricao(valor, caminho.name)


## 3. Persistência opcional

O resultado permanece apenas na memória quando o caminho de saída é `None`. Se um arquivo `.json` for configurado, a pasta é criada e o contrato completo é gravado em UTF-8.

In [ ]:
def salvar_resultado(
    resultado: dict[str, Any], caminho_saida: str | Path | None
) -> Path | None:
    """Grava o JSON somente quando um caminho de saída é informado."""
    if caminho_saida is None:
        return None
    if not isinstance(resultado, dict):
        raise TypeError("O resultado deve ser um dicionário.")
    caminho = Path(caminho_saida)
    if caminho.suffix.casefold() != ".json":
        raise ValueError("O arquivo de saída deve usar a extensão .json.")
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(
        json.dumps(resultado, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return caminho


## 4. Contrato integrado

`analisar_transcricao` é a única interface que a pessoa usuária precisa conhecer. Ela valida a entrada, coordena os módulos e registra para cada indicador se houve uso de modelo ou fallback.

In [ ]:
def analisar_transcricao(transcricao: str, modo: str = "auto") -> dict[str, Any]:
    """Analisa uma transcrição e devolve o contrato comercial da Sprint 4."""
    if not isinstance(transcricao, str):
        raise TypeError("A transcrição deve ser uma string.")
    if not transcricao.strip():
        raise ValueError("A transcrição não pode estar vazia.")
    if modo not in SUPPORTED_MODES:
        validos = ", ".join(sorted(SUPPORTED_MODES))
        raise ValueError(f"Modo inválido: {modo!r}. Use um de: {validos}.")

    produtos_candidatos, product_mode, product_reason = _analyze_products(
        transcricao, modo
    )
    sentimento, sentiment_mode, sentiment_reason = _analyze_sentiment(transcricao, modo)
    risco_churn, churn_mode, churn_reason = _analyze_churn(transcricao, modo)
    oportunidade, opportunity_mode, opportunity_reason = _analyze_opportunity(
        transcricao, modo
    )
    principais_termos = _extract_key_terms(transcricao)
    recomendacao = _recommend_action(
        produtos_candidatos, sentimento, risco_churn, oportunidade
    )
    fallback_reasons = {}
    if product_reason:
        fallback_reasons["products"] = product_reason
    if sentiment_reason:
        fallback_reasons["sentiment"] = sentiment_reason
    if churn_reason:
        fallback_reasons["churn"] = churn_reason
    if opportunity_reason:
        fallback_reasons["opportunity"] = opportunity_reason

    return {
        "schema_version": "1.0",
        "transcricao_original": transcricao,
        "produto_identificado": (
            produtos_candidatos[0]["product"] if produtos_candidatos else None
        ),
        "produtos_candidatos": produtos_candidatos,
        "sentimento": sentimento,
        "risco_churn": risco_churn,
        "oportunidade_comercial": oportunidade,
        "principais_termos": principais_termos,
        "recomendacao_acao": recomendacao,
        "analysis_mode": {
            "requested": modo,
            "components": {
                "products": product_mode,
                "sentiment": sentiment_mode,
                "churn": churn_mode,
                "opportunity": opportunity_mode,
            },
            "fallback_reasons": fallback_reasons,
        },
    }


## 5. Configurar a entrada

Escolha `texto` para colar uma transcrição ou `arquivo` para ler `.txt`/`.json`. Em JSON, ajuste `CAMPO_JSON` ao nome que contém o texto, como `transcricao` ou `ANON_TRANSCRICAO`. Deixe `ARQUIVO_SAIDA=None` para não gravar o resultado.

In [ ]:
FONTE_ENTRADA = "texto"  # texto ou arquivo
TRANSCRICAO_DIRETA = "Cole aqui a transcrição original da reunião."
ARQUIVO_ENTRADA = None  # exemplo: Path("data/minha_reuniao.txt")
CAMPO_JSON = "transcricao"
MODO_ANALISE = "auto"  # auto, full ou fallback
ARQUIVO_SAIDA = None  # exemplo: Path("reports/minha_analise.json")


## 6. Executar e revisar

O JSON mantém a transcrição original e os metadados necessários para auditar os indicadores. Nenhuma ação comercial deve ocorrer antes da revisão humana.

In [ ]:
if FONTE_ENTRADA == "texto":
    transcricao = carregar_transcricao(texto=TRANSCRICAO_DIRETA)
elif FONTE_ENTRADA == "arquivo":
    transcricao = carregar_transcricao(
        arquivo=ARQUIVO_ENTRADA, campo_json=CAMPO_JSON
    )
else:
    raise ValueError("FONTE_ENTRADA deve ser 'texto' ou 'arquivo'.")

resultado = analisar_transcricao(transcricao, modo=MODO_ANALISE)
caminho_salvo = salvar_resultado(resultado, ARQUIVO_SAIDA)
print(json.dumps(resultado, ensure_ascii=False, indent=2))
if caminho_salvo is not None:
    print(f"Resultado salvo em: {caminho_salvo}")
